<a href="https://colab.research.google.com/github/kbook6/tamu-engy604/blob/main/604groupproject_partB_kat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ENGY 604 Group Project -- Group 2
## Part B

### Minimize the total system costs for a 24-hour period with variable lighting demand.

In [ ]:
# objectives:
# create a new model with the same processes and resources
# set operating horizon to one day
# update the model to have the same refrigeration & heating demands, but variable lighting demand

In [ ]:
!pip install energiapy

In [ ]:
!apt-get install -y glpk-utils
!pip install pyomo pandas numpy

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
glpk-utils is already the newest version (5.0-1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [ ]:
import energia
from energia import *
from pyomo.environ import Objective, Constraint

In [ ]:
print(energia.__version__)

2.1.5


In [ ]:
import pandas as pd
import io

## Model Set-Up

In [ ]:
print("=== CREATING NEW MODEL FOR PART B ===")
m_b = Model()

### Set up multi-period operation ###
#m_b.h = Periods(label='Hour', size=24) # 24 hour operation (1 day)
#m_b.operating_horizon = m_b.h
m_b.h = Periods()
m_b.operating_horizon = m_b.h * 24
print(f"Set up {len(m_b.operating_horizon)} hourly periods")
days_per_year = 365

### set up currency ###
m_b.usd = Currency()

=== CREATING NEW MODEL FOR PART B ===
Set up 1 hourly periods


In [ ]:
### import the demand schedule from Canvas ###
from google.colab import files
uploaded = files.upload()

Saving Lighting energy demand for scheduling.xlsx to Lighting energy demand for scheduling (1).xlsx


In [ ]:
### create a list of the demand schedule ###
# Read the uploaded file
file_name = list(uploaded.keys())[0]

# Read CSV into pandas DataFrame
df = pd.read_excel(io.BytesIO(uploaded[file_name]))

# Extract just the column w/ numbers
demand_list = df.iloc[:, 1].tolist()  # Second column values as list

print("Values list:", demand_list)
print("Length:", len(demand_list))

Values list: [91, 88, 87, 88, 92, 101, 112, 123, 132, 141, 150, 155, 159, 163, 165, 166, 168, 176, 183, 180, 172, 165, 156, 147]
Length: 24


In [ ]:
### resources ###
m_b.declare(Resource, ['bm','ng','elec_g','solar','wind','heat','electricity','co2', 'lighting', 'heating', 'refrigeration'])
print(m_b.resources)

# consumed
m_b.bm.consume == True
m_b.ng.consume == True
m_b.elec_g.consume == True
m_b.solar.consume == True
m_b.wind.consume == True

# produced
m_b.refrigeration.release == True
m_b.lighting.release == True
m_b.heating.release == True
m_b.co2.release == True

⚖  Initiated bm balance in (l0, operating_horizon)                          ⏱ 0.0004 s


[bm, ng, elec_g, solar, wind, heat, electricity, co2, lighting, heating, refrigeration]


INFO:energia:⚖  Initiated bm balance in (l0, operating_horizon)                          ⏱ 0.0004 s
⚖  Initiated ng balance in (l0, operating_horizon)                          ⏱ 0.0005 s
INFO:energia:⚖  Initiated ng balance in (l0, operating_horizon)                          ⏱ 0.0005 s
⚖  Initiated elec_g balance in (l0, operating_horizon)                      ⏱ 0.0026 s
INFO:energia:⚖  Initiated elec_g balance in (l0, operating_horizon)                      ⏱ 0.0026 s
⚖  Initiated solar balance in (l0, operating_horizon)                       ⏱ 0.0004 s
INFO:energia:⚖  Initiated solar balance in (l0, operating_horizon)                       ⏱ 0.0004 s
⚖  Initiated wind balance in (l0, operating_horizon)                        ⏱ 0.0004 s
INFO:energia:⚖  Initiated wind balance in (l0, operating_horizon)                        ⏱ 0.0004 s
⚖  Initiated refrigeration balance in (l0, operating_horizon)               ⏱ 0.0004 s
INFO:energia:⚖  Initiated refrigeration balance in (l0, operating

In [ ]:
### setting up processes ###
m_b.st = Process()
m_b.chp = Process()
m_b.pv = Process()
m_b.wf = Process()
m_b.grid = Process()
m_b.rf = Process()
m_b.led = Process()
m_b.spaceheater = Process()

## PRIMARIES:
# biomass ST
m_b.st(m_b.electricity) == 0.68 * -m_b.bm # bm -> electricity
m_b.st.capacity.x >= 100
m_b.st.capacity.x <= 1e6
m_b.st.capex == 250
m_b.st.opex == 15 / 8760
m_b.st.label == 'Biomass ST'
m_b.st.operate == True

# natural gas CHP
m_b.chp(m_b.electricity) == 0.44 * -m_b.ng # nat gas -> electricity
m_b.chp(m_b.heat) == 0.28 * -m_b.ng # nat gas -> heat
m_b.chp.capacity.x >= 800
m_b.chp.capacity.x <= 1e6
m_b.chp.capex == 500
m_b.chp.opex == 15 / 8760
m_b.chp.label == 'Natural Gas CHP'
m_b.chp.operate == True

# solar pv
m_b.pv(m_b.electricity) == 0.09 * -m_b.solar # solar -> electricity
m_b.pv.capacity.x >= 10
m_b.pv.capacity.x <= 300
m_b.pv.capex == 2000
m_b.pv.opex == 500 / 8760
m_b.pv.label == 'Solar PV'
m_b.pv.operate == True

# wind farm
m_b.wf(m_b.electricity) == 0.22 * -m_b.wind # wind -> electricity
m_b.wf.capacity.x >= 10
m_b.wf.capacity.x <= 500
m_b.wf.capex == 2000
m_b.wf.opex == 1200 / 8760
m_b.wf.label == 'Wind Farm'
m_b.wf.operate == True

# grid
m_b.grid(m_b.electricity) == 1 * -m_b.elec_g
m_b.grid.capacity.x >= 0 # set arbitrary lower bound
m_b.grid.capacity.x <= 1e6 # set arbitrary upper bound
m_b.grid.label == 'Grid Electricity'
m_b.grid.operate == True

## SECONDARIES
# refrigerator
m_b.rf(m_b.refrigeration) == 3.0 * -m_b.electricity
m_b.rf.capacity.x >= 0 # set arbitrary lower bound
m_b.rf.capacity.x <= 1e6 # set arbitrary upper bound
m_b.rf.capex == 70
m_b.rf.opex == 4 / 8760
m_b.rf.label == 'Refrigeration'
m_b.rf.operate == True

# LED (lighting)
m_b.led(m_b.lighting) == 0.8 * -m_b.electricity
m_b.led.capacity.x >= 0 # set arbitrary lower bound
m_b.led.capacity.x <= 1e6 # set arbitrary upper bound
m_b.led.capex == 10
m_b.led.opex == 1 / 8760
m_b.led.label == 'LED Lighting'
m_b.led.operate == True

# space heater
m_b.spaceheater(m_b.heating) == 0.85 * -m_b.heat
m_b.spaceheater.capacity.x >= 0 # set arbitrary lower bound
m_b.spaceheater.capacity.x <= 1e6 # set arbitrary upper bound
m_b.spaceheater.capex == 30
m_b.spaceheater.opex == 3 / 8760
m_b.spaceheater.label == 'Space Heater'
m_b.spaceheater.operate == True

🔗  Bound [≥] st capacity in (l0, operating_horizon)                         ⏱ 0.0007 s
INFO:energia:🔗  Bound [≥] st capacity in (l0, operating_horizon)                         ⏱ 0.0007 s
🔗  Bound [≤] st capacity in (l0, operating_horizon)                         ⏱ 0.0008 s
INFO:energia:🔗  Bound [≤] st capacity in (l0, operating_horizon)                         ⏱ 0.0008 s
🔗  Bound [=] usd spend in (l0, operating_horizon)                           ⏱ 0.0018 s
INFO:energia:🔗  Bound [=] usd spend in (l0, operating_horizon)                           ⏱ 0.0018 s
🔗  Bound [=] usd spend in (l0, operating_horizon)                           ⏱ 0.0008 s
INFO:energia:🔗  Bound [=] usd spend in (l0, operating_horizon)                           ⏱ 0.0008 s
🔗  Bound [≥] chp capacity in (l0, operating_horizon)                        ⏱ 0.0017 s
INFO:energia:🔗  Bound [≥] chp capacity in (l0, operating_horizon)                        ⏱ 0.0017 s
🔗  Bound [≤] chp capacity in (l0, operating_horizon)             

In [ ]:
### Cost of Resources ###
# price for grid electricity resource
price_elec_g = (36.11/277.78) * 24 # price per kilowatt hour per day
print("price of grid electricity: ", price_elec_g)
m_b.elec_g.price.prep(price_elec_g)

# price for natural gas
price_natural_gas = (8.89/277.78) * 24
print("price of natural gas: ", price_natural_gas)
m_b.ng.price.prep(price_natural_gas)

# price for biomass
price_bm = (9.72/277.78) * 24
print("price of biomass: ", price_bm)
m_b.bm.price.prep(price_bm)

# wind and solar are free
m_b.wind.price.prep(0)
m_b.solar.price.prep(0)

price of grid electricity:  3.1198790409676724
price of natural gas:  0.7680898552811579
price of biomass:  0.8398012815897474


usd.spend

In [ ]:
### demand schedule ###
# keep refrigeration and heating the same but update the schedule to one operating day
m_b.refrigeration.release == True
m_b.heating.release == True
m_b.refrigeration_min_production = Constraint(expr = m_b.refrigeration.produce >= 1000/days_per_year)
m_b.heating_min_production = Constraint(expr = m_b.heating.produce >= 100/days_per_year)

# update lighting with new demand schedule
m_b.lighting.demand.prep(demand_list)

# set lighting release
m_b.lighting.release == True
m_b.lighting_min_production = Constraint(expr = m_b.lighting.produce >= 200/days_per_year)

⚖  Updated refrigeration balance with produce(refrigeration, l0, operating_horizon) ⏱ 0.0004 s
INFO:energia:⚖  Updated refrigeration balance with produce(refrigeration, l0, operating_horizon) ⏱ 0.0004 s
🔗  Bound [≥] refrigeration produce in (l0, operating_horizon)               ⏱ 0.0030 s
INFO:energia:🔗  Bound [≥] refrigeration produce in (l0, operating_horizon)               ⏱ 0.0030 s
⚖  Updated heating balance with produce(heating, l0, operating_horizon)     ⏱ 0.0008 s
INFO:energia:⚖  Updated heating balance with produce(heating, l0, operating_horizon)     ⏱ 0.0008 s
🔗  Bound [≥] heating produce in (l0, operating_horizon)                     ⏱ 0.0046 s
INFO:energia:🔗  Bound [≥] heating produce in (l0, operating_horizon)                     ⏱ 0.0046 s
⚖  Updated lighting balance with produce(lighting, l0, operating_horizon)   ⏱ 0.0004 s
INFO:energia:⚖  Updated lighting balance with produce(lighting, l0, operating_horizon)   ⏱ 0.0004 s
🔗  Bound [≥] lighting produce in (l0, operating_h

In [ ]:
### Force CHP to produce heat (since it's the only heat producer) ###
m_b.chp_must_run = Constraint(expr=m_b.chp.capacity.x >= 1)

In [ ]:
### Locate the Processes ###
m_b.locate(m_b.st, m_b.chp, m_b.pv, m_b.wf, m_b.grid, m_b.rf, m_b.led, m_b.spaceheater)

💡  Assumed st capacity unbounded in (l0, operating_horizon)                 ⏱ 0.0002 s
INFO:energia:💡  Assumed st capacity unbounded in (l0, operating_horizon)                 ⏱ 0.0002 s
🔗  Bound [≤] st operate in (l0, operating_horizon)                          ⏱ 0.0007 s
INFO:energia:🔗  Bound [≤] st operate in (l0, operating_horizon)                          ⏱ 0.0007 s
💡  Assumed st operate bounded by capacity in (l0, operating_horizon)        ⏱ 0.0031 s
INFO:energia:💡  Assumed st operate bounded by capacity in (l0, operating_horizon)        ⏱ 0.0031 s
⚖  Initiated electricity balance in (l0, operating_horizon)                 ⏱ 0.0005 s
INFO:energia:⚖  Initiated electricity balance in (l0, operating_horizon)                 ⏱ 0.0005 s
🔗  Bound [=] electricity produce in (l0, operating_horizon)                 ⏱ 0.0046 s
INFO:energia:🔗  Bound [=] electricity produce in (l0, operating_horizon)                 ⏱ 0.0046 s
⚖  Updated bm balance with expend(bm, l0, operating_horizon, oper

In [ ]:
### adding an integer cut ###
m_b.integer_cut = m_b.st.capacity.x + m_b.chp.capacity.x + m_b.pv.capacity.x + m_b.wf.capacity.x + m_b.grid.capacity.x <= 1

In [ ]:
### optimize total system cost ###
m_b.usd.spend.opt()

🧭  Mapped samples for spend (usd, l0, operating_horizon, capacity, st) ⟺ (usd, l0, operating_horizon) ⏱ 0.0016 s
INFO:energia:🧭  Mapped samples for spend (usd, l0, operating_horizon, capacity, st) ⟺ (usd, l0, operating_horizon) ⏱ 0.0016 s
🧭  Mapped samples for spend (usd, l0, operating_horizon, operate, st) ⟺ (usd, l0, operating_horizon) ⏱ 0.0010 s
INFO:energia:🧭  Mapped samples for spend (usd, l0, operating_horizon, operate, st) ⟺ (usd, l0, operating_horizon) ⏱ 0.0010 s
🧭  Mapped samples for spend (usd, l0, operating_horizon, capacity, chp) ⟺ (usd, l0, operating_horizon) ⏱ 0.0007 s
INFO:energia:🧭  Mapped samples for spend (usd, l0, operating_horizon, capacity, chp) ⟺ (usd, l0, operating_horizon) ⏱ 0.0007 s
🧭  Mapped samples for spend (usd, l0, operating_horizon, operate, chp) ⟺ (usd, l0, operating_horizon) ⏱ 0.0009 s
INFO:energia:🧭  Mapped samples for spend (usd, l0, operating_horizon, operate, chp) ⟺ (usd, l0, operating_horizon) ⏱ 0.0009 s
🧭  Mapped samples for spend (usd, l0, operat

Restricted license - for non-production use only - expires 2027-11-29
Read MPS format model from file Program(m).mps
Reading time = 0.02 seconds
PROGRAM(M): 70 rows, 68 columns, 169 nonzeros


📝  Generated gurobipy model. See .formulation                               ⏱ 0.1045 s
INFO:gana:📝  Generated gurobipy model. See .formulation                               ⏱ 0.1045 s


Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (linux64 - "Ubuntu 22.04.4 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 70 rows, 68 columns and 169 nonzeros (Min)
Model fingerprint: 0xa468863a
Model has 1 linear objective coefficients
Variable types: 60 continuous, 8 integer (8 binary)
Coefficient statistics:
  Matrix range     [1e-04, 1e+06]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e-01, 3e+00]
Presolve removed 59 rows and 53 columns
Presolve time: 0.03s
Presolved: 11 rows, 15 columns, 33 nonzeros
Variable types: 10 continuous, 5 integer (5 binary)
Found heuristic solution: objective 400000.00000
Found heuristic solution: objective 0.0000000

Root relaxation: cutoff, 0 iterations, 0.00 seconds (0.00 work units)

Explored 1 nodes (0 simplex iterations) in 0.09 seconds (0.00 work units)
Thread count was 2

📝  Generated Solution object for Program(m). See .solution                  ⏱ 0.0011 s
INFO:gana:📝  Generated Solution object for Program(m). See .solution                  ⏱ 0.0011 s
✅  Program(m) optimized using gurobi. Display using .output()               ⏱ 0.2202 s
INFO:gana:✅  Program(m) optimized using gurobi. Display using .output()               ⏱ 0.2202 s


In [ ]:
m_b.output()
# output automatically goes to the first solution listed, which is zero

# Solution for Program(m)

<br><br>

## Objective

<IPython.core.display.Math object>

<br><br>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [ ]:
### obtain info for the second solution ###
# our constraints are not working to output nonzero solutions
# instead, use gurobi to print the output variables:

## set up gurobi
from gurobipy import GRB

gm = m_b.formulation  # underlying Gurobi model inside Energia
gm.setParam('PoolSearchMode', 2)
gm.setParam('PoolSolutions', 10)
gm.optimize()


Set parameter PoolSearchMode to value 2
Set parameter PoolSolutions to value 10
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (linux64 - "Ubuntu 22.04.4 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Non-default parameters:
PoolSearchMode  2

Optimize a model with 70 rows, 68 columns and 169 nonzeros (Min)
Model fingerprint: 0xa468863a
Model has 1 linear objective coefficients
Variable types: 60 continuous, 8 integer (8 binary)
Coefficient statistics:
  Matrix range     [1e-04, 1e+06]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e-01, 3e+00]
Presolved: 11 rows, 15 columns, 33 nonzeros

Continuing optimization...


Explored 1 nodes (0 simplex iterations) in 0.11 seconds (0.00 work units)
Most recent optimization runtime was 0.01 seconds (0.00 work units)
Thread count was 2 (of 2 available processors)

Solution count 2: 0 400000 
N

In [ ]:
## select for the second solution from gurobi
if gm.SolCount > 1:
    gm.setParam(GRB.Param.SolutionNumber, 1)  # 0-based index → 1 = second solution
    print("Objective value for 2nd solution:", gm.PoolObjVal)
else:
    print("Only one solution found.")

Objective value for 2nd solution: 400000.0


In [ ]:
## print the information linked to the second output solution
gm = m_b.formulation
gm.setParam('SolutionNumber', 1)  # 0-based index: 1 = second solution

vars_ = gm.getVars()
vals_ = gm.getAttr('Xn', vars_)

print("=== Second solution (objective =", gm.PoolObjVal, ") ===")
for v, val in zip(vars_, vals_):
    if abs(val) > 1e-6:
        print(f"{v.VarName} = {val}")

=== Second solution (objective = 400000.0 ) ===
V5 = 2.73972602739726
V6 = 0.547945205479452
V7 = 0.273972602739726
V14 = 800.0
X15 = 1.0
V16 = 400000.0
X33 = 1.0
X38 = 1.0
X43 = 1.0
V47 = 2.73972602739726
V48 = 0.273972602739726
V49 = 0.547945205479452
V67 = 400000.0


In [ ]:
### checking math models ###
m_b.show()

# Mathematical Program for Program(m)

<br><br>

## Index Sets

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

### Mapping Constraint Sets

<IPython.core.display.Math object>

###  Function Sets

<IPython.core.display.Math object>

In [ ]:
# --------------------------------------------------------------------------------------------------------------------------------- #
# --------------------------------------------------------------------------------------------------------------------------------- #
# --------------------------------------------------------------------------------------------------------------------------------- #